# tmpl

> The template layer shared by claudedojo and codexdojo

In [ ]:
#| default_exp tmpl

#| export
`claudedojo` and `codexdojo` both capture a clean practice round and add it to later sessions. This module provides their shared prompts, completion receipts, storage, and validation. It also selects the round's boundaries, refreshes recorded outputs, and reads launch configuration.

Claude Code stores transcript records. Codex stores Responses items. Each backend supplies the conversion between its records and the shared checks. Backends also manage their own transport and capture process.

In [ ]:
#| export
import json, os, re, subprocess, sys, tempfile, tomllib
from importlib.resources import files
from fastcore.utils import *
from fastcore.script import call_parse
from clikernel.cli import _MARKER as _CLIK_MARKER
from aidialog.ipynb import read_ipynb, write_ipynb
from aidialog.hist import reply2dlg, dlg2reply, _parse_call
from aidialog.dialog import code_output, prompt_output

In [ ]:
from fastcore.test import *
import tempfile, shutil

## The shared script

Both hosts use `TMPL_PROMPT` to open a template and `APPEND_PROMPT` after compaction. The capture child follows `CAPTURE_SCRIPT`. `GATE_FORBID` lists phrases from that script that must not appear in the template's visible narration.

In [ ]:
#| export
TMPL_PROMPT = "Bootstrap and complete the dojo and tell me when you're ready."
APPEND_PROMPT = "We've compacted - run the dojo round again and tell me when you're ready."
GATE_FORBID = ('recording a demo', 'these notes')
CAPTURE_SCRIPT = (files('llmdojo')/'dojo_data'/'capture_prompt.md').read_text()

## Completion receipts

A clean score prints a four-character hexadecimal completion ID. `find_cid` returns the first ID in an iterable of record texts. Each backend supplies text through its own extractor. No match returns `None`.

The store records the dojo version alongside the receipt. Launch-time registration uses both values to decide whether the completed round still counts.

In [ ]:
#| export
def find_cid(
    txts, # Record texts, in order
):
    "The completion id receipt in a clean round's texts"
    return first(m[1] for t in txts if (m := re.search(r'Completion id: ([0-9a-f]{4})', t)))

def _dojo_v():
    from llmdojo.dojo import dojo_version
    return dojo_version()

In [ ]:
test_eq(find_cid(['strokes 8 = 8, par 8', 'Clean round. Completion id: ab12 - keep this id.']), 'ab12')
test_eq(find_cid(['no receipt here']), None)

## The template store

A store contains `template.jsonl` and `meta.json`. The first file contains the round in its backend's format. Each line is a Claude transcript record or a Codex Responses item. The store doesn't convert between those formats.

`meta.json` contains the completion ID as `cid` and the tooling version as `v`.

In [ ]:
#| export
def save_store(
    d, # Store dir
    items, # Native template items or records
    cid, # The round's completion id receipt
):
    "Write a template and its metadata to the store at `d`"
    d = Path(d)
    d.mkdir(parents=True, exist_ok=True)
    (d/'template.jsonl').write_text(''.join(json.dumps(obj2dict(x))+'\n' for x in items))
    (d/'meta.json').write_text(json.dumps(dict(cid=cid, v=_dojo_v())))

def load_store(
    d, # Store dir
):
    "The stored native items and metadata at `d`"
    d = Path(d)
    return (d/'template.jsonl').read_jsonl(), json.loads((d/'meta.json').read_text())

In [ ]:
items = [dict(role='user', content='hi'), dict(role='assistant', content='Completion id: ab12')]
store = Path(tempfile.mkdtemp())
save_store(store, items, 'ab12')
back,meta = load_store(store)
test_eq(list(back), items)
test_eq(meta, dict(cid='ab12', v=_dojo_v()))
shutil.rmtree(store)
meta

{'cid': 'ab12', 'v': '0.0.7:5'}

Both launchers call `load_reg` to load a template and register its completion ID. `dojobuild` creates the packaged stores from the canonical dialog. Updating llmdojo updates those stores. Users don't need to build a template before their first launch.

If the stored version differs from the installed tooling, `load_reg` prints a warning to stderr. This usually means the tooling version changed without a template rebuild. The loader still registers the receipt with its original version. `dojo_start(cid)` won't accept it under a different version. Keeping warnings off stdout also supports the launchers' session-ID output mode.

Registration writes to the machine's dojo state. Tests that exercise it must redirect that state directory.

In [ ]:
#| export
def load_reg(
    d, # Store dir; `default` if None
    default, # The backend's packaged store dir
    prog, # The backend's program name, for the skew warning
):
    "Load a template store, warn on version skew, and register its completion id"
    items,meta = load_store(Path(d or default))
    if meta['v'] != _dojo_v(): print(f"{prog}: template built under {meta['v']} but installed tooling is {_dojo_v()}; the baked round will not validate. Rebuild with dojobuild.", file=sys.stderr)
    from llmdojo.dojo import register_completion
    register_completion(meta['cid'], meta['v'])
    return items,meta

## Content gates

`round_gates` checks whether a capture demonstrates a clean first attempt. It requires one dealt round and one score. It rejects redos, resumes, missing clean-score output, and forbidden script phrases in the visible reply.

These checks use kernel cell sources, tool output text, and assistant text. Each backend's `is_clean` adds checks for its record format. Examples include Claude error results and Codex calls without matching results.

In [ ]:
#| export
_START_RE = r'^dojo_start\(\s*\)\s*$'

def round_gates(
    cells, # The round's kernel cell sources, in order
    outs, # Its tool output texts
    visible, # Its visible assistant text, joined
    forbid=GATE_FORBID, # Phrases that must not appear in visible text
):
    "The content problems that disqualify a round from becoming a template, whichever host played it"
    probs = []
    if (n := sum(bool(re.match(_START_RE, c)) for c in cells)) != 1: probs.append(f'{n} dojo_start calls')
    if (n := sum(bool(re.match(r'^dojo_score\(', c)) for c in cells)) != 1: probs.append(f'{n} dojo_score calls (a clean round scores once, first try)')
    if any(re.match(r'^dojo_(?:redo|resume)\(', c) for c in cells): probs.append('needed a redo')
    if not any('Clean round' in o for o in outs): probs.append('no clean score')
    if (bad := [f for f in forbid if f in visible]): probs.append(f"script leaked into visible text: {', '.join(bad)}")
    return probs

A bare `dojo_start()` deals a round. A call such as `dojo_start('ab12')` presents a previous receipt instead. The receipt form doesn't count as another dealt round:

In [ ]:
boot = ['doc(clik, pysk, edsk)', 'doc(dsk, exh, rgsk)', 'doc(acp)', 'list_pyskills()']
round_cells = ['dojo_start()', '%cd /tmp/kata', '# kata 1', 'doc(report.daily_report)', "dojo_score(bash_calls=0, report='RB7034')"]
outs = ['...', 'Clean round. Completion id: ab12']
test_eq(round_gates(boot+round_cells, outs, 'OK ready.'), [])
test_eq(round_gates(["dojo_start('ab12')"]+boot+round_cells, outs, 'OK ready.'), [])

The returned list reports every failed check. These examples cover another deal, another score, a redo, missing clean-score text, and script leakage.

In [ ]:
test_eq(round_gates(['dojo_start()']+round_cells, outs, ''), ['2 dojo_start calls'])
test_eq(round_gates(round_cells+['dojo_score(1)'], outs, ''), ['2 dojo_score calls (a clean round scores once, first try)'])
test_eq(round_gates(round_cells+['dojo_redo(3)'], outs, ''), ['needed a redo'])
test_eq(round_gates(round_cells, ['...'], ''), ['no clean score'])
round_gates(round_cells, outs, 'as these notes say')

['script leaked into visible text: these notes']

## The round's shape

`capture_slice` selects a round from a list of kernel cell sources. It finds the last bare `dojo_start()`, then the last bootstrap `doc(clik, pysk, edsk)` before it. The slice ends at the first `dojo_score` after that start. Both returned indices are inclusive.

These calls identify the boundaries without extra markers in the transcript. Each backend maps its calls and results to cell sources before selecting the records.

In [ ]:
#| export
def capture_slice(
    cells, # Kernel cell sources of a conversation's calls, in order
):
    "Return inclusive bootstrap-to-score indices for the last dealt round."
    starts = [i for i,c in enumerate(cells) if re.match(_START_RE, c)]
    if not starts: raise ValueError('no dojo_start() call')
    start = starts[-1]
    boots = [i for i,c in enumerate(cells[:start]) if re.match(r'^doc\(clik,\s*pysk,\s*edsk\)\s*$', c)]
    if not boots: raise ValueError('no bootstrap doc call before dojo_start()')
    scores = [i for i,c in enumerate(cells[start:], start) if re.match(r'^dojo_score\(', c)]
    if not scores: raise ValueError('no dojo_score() after dojo_start()')
    return boots[-1], scores[0]

In [ ]:
test_eq(capture_slice(boot+round_cells), (0, 8))


The latest dealt round takes precedence over earlier rounds. If it has no score, `capture_slice` raises instead of falling back to an earlier completed round. Missing start and bootstrap calls also raise:

In [ ]:
test_eq(capture_slice(boot+round_cells+boot+round_cells), (9, 17))
test_fail(lambda: capture_slice(boot), contains='no dojo_start')
test_fail(lambda: capture_slice(round_cells), contains='no bootstrap doc call')
test_fail(lambda: capture_slice(boot+['dojo_start()']), contains='no dojo_score')

Compaction should keep the conversation's work rather than previous dojo rounds. Each backend removes whole user turns containing a round. `_turns` groups the records at user prompts, including a leading group for records before the first prompt. The backend supplies the prompt predicate and decides which groups to remove.

In [ ]:
#| export
def _turns(
    xs, # Items or records in conversation order
    isprompt, # Is this element a real user prompt?
):
    "`xs` split into user turns at each prompt (a leading group holds anything earlier)"
    xs = L(xs)
    edges = L([0, *xs.argwhere(isprompt), len(xs)])
    return L(xs[s:e] for s,e in edges.pairwise())

In [ ]:
turns = _turns(['a','p1','x','y','p2','z'], lambda x: x.startswith('p'))
test_eq(turns, [['a'],['p1','x','y'],['p2','z']])
turns

[['a'], ['p1', 'x', 'y'], ['p2', 'z']]

## Refreshing a template

A template's recorded outputs become stale when tooling docs, kata cards, or rendering change. Refresh them by replaying the existing cells. This requires no model calls. To change the round's cells or narration, capture a new round or edit the template separately.

`_replay_cells` starts a fresh `clikernel` CLI process in a scratch project. It runs the cells in order using the normal kernel startup, magics, and rendering. By default, the dojo uses the machine's real state directory. Replaying a score registers its content-derived completion receipt there.

Stored templates use `/tmp/dojo` instead of a machine-specific run directory. Refreshing replaces that spelling with the local run directory before execution. It converts the path back before writing the template. The same round can then produce identical files across refreshes. Outputs still depend on installed tooling and environment, including the pyskill catalog.

In [ ]:
#| export
def _replay_cells(
    cells, # Kernel cell sources, run in order
    cwd, # Directory to start the kernel in
    env=None, # Extra environment entries for the kernel process
):
    "Each cell's rendered response from a fresh `clikernel` CLI at `cwd`"
    p = subprocess.Popen(['clikernel'], cwd=str(cwd), env=os.environ|(env or {}), text=True,
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    def upto(stop):
        res = []
        for line in p.stdout:
            if (line := line.rstrip('\n')) == stop: return res
            res.append(line)
        raise RuntimeError(f'clikernel exited during replay; last lines: {res[-5:]}')
    upto(_CLIK_MARKER)
    delim = p.stdout.readline().rstrip('\n')
    outs = []
    for c in cells:
        p.stdin.write(f'--\n{c}\n{delim}\n')
        p.stdin.flush()
        if (ack := p.stdout.readline().rstrip('\n')) != '.': raise RuntimeError(f'expected ack, got {ack!r}')
        outs.append('\n'.join(upto(delim)))
    p.stdin.write('exit\n')
    p.stdin.flush()
    p.wait()
    return outs

In [ ]:
rd = Path(tempfile.mkdtemp())
_replay_cells(['1+1', "print('hi')", 'x = 3'], rd)

['2', 'hi', '']

`refresh_template` replays the kernel calls in a template reply and replaces their recorded outputs. Each call must use `mcp__clikernel__exec` or Codex's `tools.mcp__clikernel__exec` with a nonempty `code` string. Other calls raise before replay starts. An empty result uses the host's no-output message.

The function checks the refreshed round with `round_gates` before writing. A failed check leaves the destination unchanged. If the round no longer passes, review its cells rather than refreshing the same failure into the store.

In [ ]:
#| export
DOJO_CANON = '/tmp/dojo'

def canon_tmpl(
    dlg, # A template dialog whose reply holds a round
    canon=DOJO_CANON, # Canonical spelling for the round's run-dir path
):
    "Find the run directory in the reply's `%cd` cell and replace it throughout with `canon`."
    pm = dlg.messages[0]
    txt = pm.ai_res
    rd = first(re.findall(r"%cd ([^\s'\"\\]+)", txt))
    if rd and rd != canon: pm.output = prompt_output(txt.replace(rd, canon))
    return dlg

def refresh_template(
    src, # Path to a template dialog .ipynb
    dst=None, # Where to write the refreshed dialog; `src` if None
):
    "Replay template cells and replace their outputs. Validate, canonicalize, write, and return the dialog."
    from llmdojo.dojo import _run_dir
    dlg = read_ipynb(str(src))
    pmsg = dlg.messages[0]
    sub = reply2dlg(pmsg)
    codes = [m for m in sub.messages if m.msg_type=='code']
    calls = [_parse_call(m.content) for m in codes]
    for m,c in zip(codes, calls):
        valid = (c and c[0] in ('mcp__clikernel__exec', 'tools.mcp__clikernel__exec')
            and isinstance(c[1].get('code'), str) and c[1]['code'].strip())
        if not valid: raise ValueError(f'not a kernel call: {m.content}')
    cells = [c[1]['code'].replace(DOJO_CANON, str(_run_dir())) for c in calls]   # localize: the round plays in the real run dir
    with tempfile.TemporaryDirectory(prefix='dojorefresh_') as td:
        proj = Path(td)
        (proj/'pyproject.toml').write_text('[project]\nname = "dojo-refresh"\nversion = "0"\n')
        outs = _replay_cells(cells, proj)
    for m,(name,_),o in zip(codes, calls, outs): m.output = code_output(o or f'({name} completed with no output)')
    if probs := round_gates(cells, outs, ' '.join(m.content for m in sub.messages if m.msg_type=='note')): raise ValueError('; '.join(probs))
    pmsg.output = prompt_output(dlg2reply(sub).replace(str(_run_dir()), DOJO_CANON))   # canonicalize: the stored artifact reads the same everywhere
    write_ipynb(dlg, str(dst or src))
    return dlg

This acceptance test refreshes the packaged round twice and compares the resulting files byte for byte. It makes no model calls. By default it uses real dojo state, including completion registration. Repeated runs register the same content-derived receipt when the outputs are unchanged.

In [ ]:
td = Path(tempfile.mkdtemp())
src = files('llmdojo')/'dojo_data'/'dojo_template.ipynb'
r1 = refresh_template(src, td/'r1.ipynb')
r2 = refresh_template(src, td/'r2.ipynb')
test_eq((td/'r1.ipynb').read_bytes(), (td/'r2.ipynb').read_bytes())
assert DOJO_CANON in r1.messages[0].ai_res

In [ ]:
from llmdojo.dojo import _run_dir

In [ ]:
assert str(_run_dir()) not in r1.messages[0].ai_res
shutil.rmtree(td)
len(r1.messages)

1

## Launch config

The launchers start `claude` or `codex` themselves. This lets them supply standing arguments, such as a team's system-prompt files. A command that returns a session ID alone can't supply those arguments.

`launch_config` reads `<app>_args` from `$XDG_CONFIG_HOME/<app>dojo/config.toml`. Without `XDG_CONFIG_HOME`, it uses `~/.config`. It expands `~` in each argument.

In [ ]:
#| export
def launch_config(
    app, # Host tool name ('claude' or 'codex'): config at `$XDG_CONFIG_HOME/<app>dojo/config.toml`
    cfg=None, # Config file path, overriding the `app` default
):
    "Extra `app` args from the config file's `<app>_args` list, each `~`-expanded"
    if cfg is None: cfg = Path(os.environ.get('XDG_CONFIG_HOME') or '~/.config').expanduser()/f'{app}dojo'/'config.toml'
    if not Path(cfg).exists(): return []
    return [os.path.expanduser(o) for o in tomllib.loads(Path(cfg).read_text()).get(f'{app}_args', [])]

A missing config file contributes no extra arguments. The launcher can still resume a session without configuration:

In [ ]:
cdir = Path(tempfile.mkdtemp())
test_eq(launch_config('claude', cdir/'config.toml'), [])
(cdir/'config.toml').write_text('claude_args = ["--system-prompt-file", "~/prompts/sysp.md"]')
args = launch_config('claude', cdir/'config.toml')
assert '~' not in args[1]
args

['--system-prompt-file', '/Users/jhoward/prompts/sysp.md']

## The dojobuild CLI

Run `dojobuild` to refresh the canonical template dialog and compile both backend stores. Refreshing checks the round before writing the updated dialog. The launchers handle capture and session launch separately.

Use `--claude` or `--codex` to compile one store without replaying the cells. This is the build step after reviewing a capture or hand-edited template. You can supply a dialog path. Otherwise the command uses the packaged `dojo_data/dojo_template.ipynb`.

In [ ]:
#| export
@call_parse(pos=['dialog'])
def main(
    dialog:str=None, # Template dialog path; the packaged dojo_data/dojo_template.ipynb where omitted
    claude:bool=False, # Build only the Claude store from the dialog, without refreshing
    codex:bool=False, # Build only the Codex store, without refreshing
):
    "Refresh the canonical template dialog (replaying its cells through a fresh clikernel), then build both stores"
    src = dialog or files('llmdojo')/'dojo_data'/'dojo_template.ipynb'
    from llmdojo import claudedojo, codexdojo
    if not (claude or codex):
        refresh_template(src)
        print(f'refreshed: {src}')
    if not codex:
        claudedojo.build_template(src)
        print(f'built: claude store ({claudedojo.TMPL_DIR})')
    if not claude:
        codexdojo.build_template(src)
        print(f'built: codex store ({codexdojo.TMPL_DIR})')

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()